# Real-occlusion test set and duplicate check

Two inputs of the evaluation, made once before the main training:
1. **Real occlusions**: faces that the cleaning step rejected only because of glasses / hats / hands or objects
   (every other quality check passed). They keep their human expression labels and were never used for training:
   a test of occlusions that the synthetic occluder never produced (`make_real_occlusion_set.py`).
2. **Duplicates**: ExpW, AffectNet and RAF-DB were all collected from the web, so a test face can also be a train face
   of another dataset (or of the same one). Train faces that duplicate an evaluation face are listed in
   `exclude_train.csv` and left out of the training and of the emotion probe (`find_duplicates.py`).

Before running:
- Settings: **Accelerator None** (CPU only), **Internet on**.
- Inputs: `clean-faces`, `clean-faces-rafdb`, `expw-crops`, `affectnet_1`, `rafdb-crops`.

Then *Save Version → Save & Run All* and create a **Private** dataset `eval-extras` from the output of the finished
version. Add `eval-extras` as input of notebooks 04 (evaluation) and 06 (training).

In [ ]:
import os, glob
os.chdir('/kaggle/working')
!rm -rf /tmp/smirk && git clone -q --depth 1 -b occlusion-aware https://github.com/THUONGVIP09/smirk.git /tmp/smirk
!cd /tmp/smirk && git log --oneline -1
SCRIPTS = '/tmp/smirk/datasets/preprocess_scripts'

def dataset_of(path):
    path = os.path.relpath(path, '/kaggle/input').lower()
    return next((k for k in ('expw', 'affectnet', 'rafdb') if k in path), None)

# crops: meta.csv (labels) + images/
CROPS = {}
for meta in glob.glob('/kaggle/input/**/meta.csv', recursive=True):
    folder = os.path.dirname(meta)
    if dataset_of(folder) and os.path.isdir(os.path.join(folder, 'images')):
        CROPS[dataset_of(folder)] = folder
CLEAN_DIRS = sorted(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/clean_meta.csv', recursive=True))
# landmarks and quality of EVERY face (also the rejected ones), written by the cleaning notebook
ALL = {}
for quality in glob.glob('/kaggle/input/**/*_quality.csv', recursive=True):
    name = os.path.basename(quality)[:-len('_quality.csv')]
    landmarks = os.path.join(os.path.dirname(quality), f'{name}_landmarks.npz')
    if name in CROPS and os.path.exists(landmarks):
        ALL[name] = (landmarks, quality)
print('crops:', CROPS)
print('clean dirs:', CLEAN_DIRS)
print('landmarks / quality of all faces:', ALL)
assert CROPS and CLEAN_DIRS and ALL, 'add clean-faces, clean-faces-rafdb and the *-crops datasets as input'
missing = set(CROPS) - set(ALL)
assert not missing, f'no <name>_landmarks.npz / <name>_quality.csv for {missing}: add the matching clean-faces dataset'
OUT = '/kaggle/working/eval_extras'

## 1. Real-occlusion test set
Same quality thresholds as the cleaning (notebook 02, CELL 9); a face is taken when it fails only the occlusion checks.

In [ ]:
specs = ' '.join(f'--dataset {n}={CROPS[n]}/meta.csv,{landmarks},{quality}' for n, (landmarks, quality) in ALL.items())
!python {SCRIPTS}/make_real_occlusion_set.py {specs} --out_dir {OUT}/real_occlusion

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, cv2
real = pd.read_csv(f'{OUT}/real_occlusion/real_occluded_meta.csv', dtype={'name': str})
print(pd.crosstab([real['dataset'], real['expression']], real['occlusion'], margins=True))

# a few faces of every dataset and occlusion type: check that they are really occluded
groups = list(real.groupby(['dataset', 'occlusion']))
fig, axes = plt.subplots(len(groups), 8, figsize=(16, 2.2 * len(groups)), squeeze=False)
for row, ((dataset, occlusion), group) in zip(axes, groups):
    for ax in row: ax.axis('off')
    for ax, (_, r) in zip(row, group.sample(min(8, len(group)), random_state=0).iterrows()):
        ax.imshow(cv2.imread(f"{CROPS[dataset]}/images/{r['name']}.png")[..., ::-1])
        ax.set_title(f"{dataset} {occlusion}\n{r['expression']}", fontsize=8)
plt.tight_layout(); plt.show()

## 2. Duplicates between the evaluation faces (val / test / real occlusions) and the train faces
Candidates by perceptual hash of the face region (also mirrored), confirmed by the correlation of the aligned faces
(`--min_correlation 0.9`). The pairs shown below (around the threshold) tell whether the threshold is right: pairs above
it must be the same photo; if different photos pass, raise it (re-running this cell takes a few minutes).

In [ ]:
args = ' '.join([f'--meta_dir "{d}"' for d in CLEAN_DIRS] + [f'--meta_dir {OUT}/real_occlusion'] +
                [f'--image_dir "{n}={CROPS[n]}/images"' for n in CROPS])
!python {SCRIPTS}/find_duplicates.py {args} --out_dir {OUT}/duplicates --min_correlation 0.9 --num_processes 4

In [ ]:
pairs = pd.read_csv(f'{OUT}/duplicates/duplicates.csv', dtype={'eval_name': str, 'train_name': str})
print('duplicate pairs (eval dataset / split -> train dataset):')
print(pd.crosstab([pairs['eval_dataset'], pairs['eval_split']], pairs['train_dataset'], values=pairs['duplicate'], aggfunc='sum').fillna(0).astype(int))

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(pairs['correlation'], bins=60, color='#2a78d6')
ax.axvline(0.9, color='#52514e', linestyle='--', linewidth=1)
ax.set_title('Correlation of the candidate pairs (dashed: threshold)', loc='left', fontsize=11)
ax.spines[['top', 'right']].set_visible(False); plt.show()

def show_pairs(rows, title):
    if len(rows) == 0:
        return print(title, ': none')
    fig, axes = plt.subplots(2, len(rows), figsize=(2 * len(rows), 4.4), squeeze=False)
    for k, (_, r) in enumerate(rows.iterrows()):
        for j, (d, n) in enumerate([(r['eval_dataset'], r['eval_name']), (r['train_dataset'], r['train_name'])]):
            axes[j, k].imshow(cv2.imread(f'{CROPS[d]}/images/{n}.png')[..., ::-1]); axes[j, k].axis('off')
            axes[j, k].set_title(f"{d} {r['eval_split'] if j == 0 else 'train'}" + (f"\n{r['correlation']:.3f}" if j == 0 else ''), fontsize=8)
    fig.suptitle(title, x=0.01, ha='left'); plt.tight_layout(); plt.show()

show_pairs(pairs[pairs['duplicate']].sample(min(8, int(pairs['duplicate'].sum())), random_state=0), 'random duplicates')
show_pairs(pairs[pairs['duplicate']].tail(8), 'least similar duplicates (just above the threshold)')
show_pairs(pairs[~pairs['duplicate']].head(8), 'most similar non-duplicates (just below the threshold)')

## 3. Save
*Save Version → Save & Run All*, then a **Private** dataset `eval-extras` from the output of the finished version:
`real_occlusion/` (`real_occluded_meta.csv`, `landmarks.npz`) and `duplicates/` (`exclude_train.csv`, `duplicates.csv`,
`preview.jpg`). Notebooks 04 and 06 find them automatically.